# rotation-matrix-3d-y-axis — ex4: verify rotation composition R(α)·R(β) = R(α+β)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five rotation-matrix patterns that ramp from `cos/sin` → matrix assembly → rotate-a-vector → composition law → rotate-a-batch. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `rotation-matrix-3d-y-axis`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** For a right-hand rotation by `θ` about the Y axis:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```

**Why the middle row is `[0, 1, 0]`.** The Y axis is the rotation axis — anything along Y stays where it is. The X-Z plane is what gets rotated.

**Right-hand convention.** Looking down the +Y axis, the rotation goes counter-clockwise: +X → -Z, +Z → +X.

**Acting on vectors.**
- Column-vector: `v' = R @ v` (input shape `(3,)`).
- Batch of row-vectors: `points' = points @ R.T` (input shape `(N, 3)`).

**Composition.** `R_y(α) @ R_y(β) = R_y(α + β)` — rotations about a single axis add angles.

### Exercise 4 — verify rotation composition R(α)·R(β) = R(α+β)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply matrix multiplication to demonstrate the group property of rotations about a single axis.
> Keywords: composition, group-property, matmul-check
> ```

**KCs targeted:** `rotation-composes-on-axis`

Implement `ex4_compose_rotations(alpha, beta)` to compute the product `R_y(α) @ R_y(β)` and return it as a `(3, 3)` tensor.

The test then independently checks that the returned matrix equals `R_y(α + β)`. This is the 1-parameter group property: rotations about the same axis add their angles.

Use your Y rotation matrix construction from Exercise 2.

In [ ]:
def ex4_compose_rotations(alpha: Tensor, beta: Tensor) -> Tensor:
    """Return R_y(alpha) @ R_y(beta). Should equal R_y(alpha + beta)."""
    raise NotImplementedError()


def _test_ex4():
    import math
    def R_y(theta):
        c, s = math.cos(theta), math.sin(theta)
        return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])

    alpha = t.tensor(0.3)
    beta = t.tensor(1.2)
    product = ex4_compose_rotations(alpha, beta)
    assert product.shape == (3, 3), f'expected (3,3), got {product.shape}'
    # Independent ground truth: R(α + β).
    expected = R_y(alpha.item() + beta.item())
    assert t.allclose(product, expected, atol=1e-5), f'composition broken:\n{product}\nvs\n{expected}'

    # Symmetric case: α + (-α) = 0 → identity.
    out_zero = ex4_compose_rotations(t.tensor(0.7), t.tensor(-0.7))
    assert t.allclose(out_zero, t.eye(3), atol=1e-5), 'R(α) @ R(-α) must be identity'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_compose_rotations(alpha: Tensor, beta: Tensor) -> Tensor:
    def R_y(theta):
        c = t.cos(theta).item()
        s = t.sin(theta).item()
        return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])
    return R_y(alpha) @ R_y(beta)
```

**Why this only works on the same axis.** `R_y(α) @ R_y(β) = R_y(α+β)` because rotations about the same axis commute. But `R_y(α) @ R_x(β) ≠ R_x(β) @ R_y(α)` in general — that's the whole point of 3-D orientation being non-commutative. Don't try to compose Euler angles by adding!

**Use of the group property.** Want to pre-compute 60 rotations 6° apart for a turntable? Compute `R_y(6°)` once and matrix-multiply it into a running accumulator. Cheaper than rebuilding from scratch each frame.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()